# cocotb sem Makefile - Exemplo VHDL

Este notebook demonstra como usar cocotb com VHDL diretamente no Jupyter Notebook.

## Requisitos
- Python 3.8+
- cocotb >= 2.0
- GHDL instalado

## 1. Verificar ambiente

In [ ]:
import shutil
import sys

print(f"Python: {sys.version}")
print(f"GHDL: {shutil.which('ghdl')}")

# Verifica cocotb
import cocotb
print(f"cocotb version: {cocotb.__version__}")

# Verifica cocotb_tools (cocotb 2.0+)
try:
    from cocotb_tools.runner import get_runner
    print("cocotb_tools.runner: OK")
except ImportError:
    from cocotb.runner import get_runner
    print("cocotb.runner: OK (versao antiga)")

## 2. Criar o design VHDL (MUX 4:1)

Vamos criar um multiplexador 4:1 de 8 bits em VHDL.

In [ ]:
hdl_code = """
-- MUX 4:1 de 8 bits em VHDL
library IEEE;
use IEEE.STD_LOGIC_1164.ALL;

entity mux4 is
    port (
        sel : in  std_logic_vector(1 downto 0);
        a   : in  std_logic_vector(7 downto 0);
        b   : in  std_logic_vector(7 downto 0);
        c   : in  std_logic_vector(7 downto 0);
        d   : in  std_logic_vector(7 downto 0);
        y   : out std_logic_vector(7 downto 0)
    );
end mux4;

architecture behavioral of mux4 is
begin
    process(sel, a, b, c, d)
    begin
        case sel is
            when "00" => y <= a;
            when "01" => y <= b;
            when "10" => y <= c;
            when "11" => y <= d;
            when others => y <= (others => '0');
        end case;
    end process;
end behavioral;
"""

# Salva o arquivo
with open("mux4.vhd", "w") as f:
    f.write(hdl_code)

print("Arquivo mux4.vhd criado!")
print(hdl_code)

## 3. Criar o testbench cocotb

O mesmo teste Python funciona para VHDL!

In [ ]:
test_code = '''
import cocotb
from cocotb.triggers import Timer
import random

@cocotb.test()
async def test_mux_sel_a(dut):
    """Testa selecao de entrada A (sel=00)"""
    dut.sel.value = 0b00
    dut.a.value = 0xAA
    dut.b.value = 0xBB
    dut.c.value = 0xCC
    dut.d.value = 0xDD
    
    await Timer(10, unit="ns")
    
    assert int(dut.y.value) == 0xAA, f"Esperado 0xAA, obtido {hex(int(dut.y.value))}"
    dut._log.info(f"sel=00: y = {hex(int(dut.y.value))} (esperado 0xAA) OK")

@cocotb.test()
async def test_mux_sel_b(dut):
    """Testa selecao de entrada B (sel=01)"""
    dut.sel.value = 0b01
    dut.a.value = 0xAA
    dut.b.value = 0xBB
    dut.c.value = 0xCC
    dut.d.value = 0xDD
    
    await Timer(10, unit="ns")
    
    assert int(dut.y.value) == 0xBB, f"Esperado 0xBB, obtido {hex(int(dut.y.value))}"
    dut._log.info(f"sel=01: y = {hex(int(dut.y.value))} (esperado 0xBB) OK")

@cocotb.test()
async def test_mux_sel_c(dut):
    """Testa selecao de entrada C (sel=10)"""
    dut.sel.value = 0b10
    dut.a.value = 0xAA
    dut.b.value = 0xBB
    dut.c.value = 0xCC
    dut.d.value = 0xDD
    
    await Timer(10, unit="ns")
    
    assert int(dut.y.value) == 0xCC, f"Esperado 0xCC, obtido {hex(int(dut.y.value))}"
    dut._log.info(f"sel=10: y = {hex(int(dut.y.value))} (esperado 0xCC) OK")

@cocotb.test()
async def test_mux_sel_d(dut):
    """Testa selecao de entrada D (sel=11)"""
    dut.sel.value = 0b11
    dut.a.value = 0xAA
    dut.b.value = 0xBB
    dut.c.value = 0xCC
    dut.d.value = 0xDD
    
    await Timer(10, unit="ns")
    
    assert int(dut.y.value) == 0xDD, f"Esperado 0xDD, obtido {hex(int(dut.y.value))}"
    dut._log.info(f"sel=11: y = {hex(int(dut.y.value))} (esperado 0xDD) OK")

@cocotb.test()
async def test_mux_random(dut):
    """Testa MUX com valores aleatorios"""
    random.seed(42)
    
    for _ in range(20):
        a = random.randint(0, 255)
        b = random.randint(0, 255)
        c = random.randint(0, 255)
        d = random.randint(0, 255)
        sel = random.randint(0, 3)
        
        dut.a.value = a
        dut.b.value = b
        dut.c.value = c
        dut.d.value = d
        dut.sel.value = sel
        
        await Timer(10, unit="ns")
        
        expected = [a, b, c, d][sel]
        assert int(dut.y.value) == expected, \
            f"sel={sel}: esperado {expected}, obtido {int(dut.y.value)}"
    
    dut._log.info("20 testes aleatorios passaram!")
'''

# Salva o arquivo de teste
with open("test_mux4_vhdl.py", "w") as f:
    f.write(test_code)

print("Arquivo test_mux4_vhdl.py criado!")

## 4. Executar a simulacao com GHDL

In [ ]:
import shutil
import sys
import os

# Adiciona diretorio atual ao path
if os.getcwd() not in sys.path:
    sys.path.insert(0, os.getcwd())

# Importa runner
try:
    from cocotb_tools.runner import get_runner
except ImportError:
    from cocotb.runner import get_runner

# Limpa build anterior
shutil.rmtree("sim_build", ignore_errors=True)

# Configura o runner para GHDL
runner = get_runner("ghdl")

# Build do design VHDL (GHDL detecta automaticamente pelo .vhd)
runner.build(
    sources=["mux4.vhd"],
    hdl_toplevel="mux4",
)

print("Build concluido!")

In [ ]:
# Executa os testes
runner.test(
    hdl_toplevel="mux4",
    test_module="test_mux4_vhdl",
)

print("\nTodos os testes VHDL passaram!")

## 5. Exemplo adicional: Comparador em VHDL

In [ ]:
# Cria o comparador em VHDL
comparator_hdl = """
-- Comparador de 8 bits em VHDL
library IEEE;
use IEEE.STD_LOGIC_1164.ALL;
use IEEE.NUMERIC_STD.ALL;

entity comparator is
    port (
        a  : in  std_logic_vector(7 downto 0);
        b  : in  std_logic_vector(7 downto 0);
        eq : out std_logic;  -- a == b
        gt : out std_logic;  -- a > b
        lt : out std_logic   -- a < b
    );
end comparator;

architecture behavioral of comparator is
begin
    eq <= '1' when a = b else '0';
    gt <= '1' when unsigned(a) > unsigned(b) else '0';
    lt <= '1' when unsigned(a) < unsigned(b) else '0';
end behavioral;
"""

with open("comparator.vhd", "w") as f:
    f.write(comparator_hdl)

# Cria o teste
comparator_test = '''
import cocotb
from cocotb.triggers import Timer

@cocotb.test()
async def test_equal(dut):
    """Testa a == b"""
    dut.a.value = 100
    dut.b.value = 100
    await Timer(10, unit="ns")
    
    assert int(dut.eq.value) == 1, "eq deveria ser 1"
    assert int(dut.gt.value) == 0, "gt deveria ser 0"
    assert int(dut.lt.value) == 0, "lt deveria ser 0"
    dut._log.info("100 == 100: OK")

@cocotb.test()
async def test_greater(dut):
    """Testa a > b"""
    dut.a.value = 200
    dut.b.value = 100
    await Timer(10, unit="ns")
    
    assert int(dut.eq.value) == 0, "eq deveria ser 0"
    assert int(dut.gt.value) == 1, "gt deveria ser 1"
    assert int(dut.lt.value) == 0, "lt deveria ser 0"
    dut._log.info("200 > 100: OK")

@cocotb.test()
async def test_less(dut):
    """Testa a < b"""
    dut.a.value = 50
    dut.b.value = 100
    await Timer(10, unit="ns")
    
    assert int(dut.eq.value) == 0, "eq deveria ser 0"
    assert int(dut.gt.value) == 0, "gt deveria ser 0"
    assert int(dut.lt.value) == 1, "lt deveria ser 1"
    dut._log.info("50 < 100: OK")

@cocotb.test()
async def test_exhaustive(dut):
    """Testa todas combinacoes de 0-15"""
    for a in range(16):
        for b in range(16):
            dut.a.value = a
            dut.b.value = b
            await Timer(1, unit="ns")
            
            assert int(dut.eq.value) == (1 if a == b else 0)
            assert int(dut.gt.value) == (1 if a > b else 0)
            assert int(dut.lt.value) == (1 if a < b else 0)
    
    dut._log.info("256 testes exaustivos passaram!")
'''

with open("test_comparator_vhdl.py", "w") as f:
    f.write(comparator_test)

print("Arquivos comparator.vhd e test_comparator_vhdl.py criados!")

In [ ]:
# Executa teste do comparador VHDL
shutil.rmtree("sim_build", ignore_errors=True)

runner = get_runner("ghdl")

runner.build(
    sources=["comparator.vhd"],
    hdl_toplevel="comparator",
)

runner.test(
    hdl_toplevel="comparator",
    test_module="test_comparator_vhdl",
)

print("\nTestes do comparador VHDL passaram!")

## 6. Limpeza

In [ ]:
import os
import shutil

# Remove arquivos gerados (descomente para limpar)
# shutil.rmtree("sim_build", ignore_errors=True)
# for f in ["mux4.vhd", "test_mux4_vhdl.py", "comparator.vhd", "test_comparator_vhdl.py", "results.xml"]:
#     if os.path.exists(f):
#         os.remove(f)

print("Para limpar, descomente as linhas acima e execute novamente.")